In [1]:
import boto3
import json
import time
import pandas as pd
import argparse 
import random

session = boto3.Session(profile_name='corp-us-east-1')
# session = boto3.Session(profile_name='default')


# Define the table name
table_name = 'prompt_hub_table'
# Create DynamoDB resource
dynamodb = session.resource('dynamodb')
table = dynamodb.Table(table_name)

In [2]:

def filter_items(field,val):
    # Scan the table to get all items where is_external is true
    response = table.scan(
        FilterExpression=f'{field} = :val',
        ExpressionAttributeValues={':val': val}
    )
    
    items = response['Items']
    print(f"found:{len(items)}")
    
    # Handle pagination if there are more items
    while 'LastEvaluatedKey' in response:
        response = table.scan(
            FilterExpression='{field} = :val',
            ExpressionAttributeValues={':val': val},
            ExclusiveStartKey=response['LastEvaluatedKey']
        )
        items.extend(response['Items'])
    return items

def update_items(items,field,val):
    # Update each item's delete_status
    updated_count = 0
    for item in items:
        # Get the primary key values from your item
        # Modify these according to your table's primary key structure
        key = {
            'id': item['id']  # Assuming 'id' is your primary key
            # Add other key attributes if you have a composite key
        }

        # Update the item
        table.update_item(
            Key=key,
            UpdateExpression=f'SET {field} = :val',
            ExpressionAttributeValues={
                ':val': val
            }
        )
        updated_count += 1

    print(f"Successfully updated {updated_count} items")
    return updated_count

def delete_items(items):
    deleted_count = 0
    
    for item in items:
        try:
            # Get the primary key values from your item
            key = {
                'id': item['id']  # Assuming 'id' is your primary key
                # Add other key attributes if you have a composite key
            }

            # Delete the item
            table.delete_item(
                Key=key
            )
            
            deleted_count += 1
            
        except Exception as e:
            print(f"Error deleting item with id {item['id']}: {str(e)}")
            continue

    print(f"Successfully deleted {deleted_count} items")
    return deleted_count

In [53]:
# # 把is_external = true都删除
# items = filter_items("is_external",True)
# update_items(items, "delete_status","deleted")

In [11]:
# 删除之前的老版本
items = filter_items("demo_version","2025v1")
delete_items(items)

found:49
Successfully deleted 49 items


49

In [12]:
# 把is_recommended = true 改成 is_recommended = False
# items = filter_items("is_recommended",False)
# update_items(items, "is_recommended",True)

## upload new template

In [13]:
def upload_to_dynamodb(table_name, json_data):
    dynamodb = session.resource('dynamodb')
    table = dynamodb.Table(table_name)

    for item in json_data:
        table.put_item(Item=item)

    print(f"Data uploaded to DynamoDB table: {table_name}")


def generate_id():
    timestamp = int(time.time() * 1000)  # Get the current timestamp in milliseconds
    random_number = str(random.randint(0, 16**6))  # Generate a random 6-digit number
    return f"{timestamp}-{random_number}"


def process_excel(filename):
    df = pd.read_excel(filename)
    time_tuple = time.localtime( time.time())
    createtime = time.strftime("%Y-%m-%d %H:%M:%S", time_tuple)
    df.dropna(inplace=True)
    df.rename(columns={'Scenario':'category',
                       'Name':'demo_name',
                       'Description':'description',
                       'Further Support':'further_support',
                       'Status':'demo_type',
                       'Simple Demo Introduction Deck':'deck_link',
                       'Demo Video Link':'demo_link',
                       'Code Repo':'code_repo_link',
                       'China Region Support':'china_region_support',
                       'Contact':'contact',
                       'Team':'team',
                       'Industries':'industry'
                       }, inplace=True)
    df['createtime'] = createtime
    df['company'] = 'default'
    df['template'] = ''
    df['id'] = df.apply(lambda x: generate_id(), axis=1)
    df['demo_version'] = '2025v1'
    df['industry'] = df.apply(lambda x: [ i  for i in x['industry'].split(',') ] ,axis=1)
    
    df_dict = json.loads(df.to_json(orient='index'))
    return  list(df_dict.values())


In [14]:


filename = "GCR GenAI Asset Hub Management-0306.xlsx"

json_data = process_excel(filename)
print(json_data)

[{'category': '翻译', 'demo_name': '基于LLM的专词翻译方案', 'description': '本专词翻译方案致力于提升专有名词的翻译质量。该方案避免了将所有专词都放入提示词的做法，提高了翻译效率。它允许用户自定义翻译规则，尤其适合专词较多的场景。', 'demo_type': 'Ready-to-Adopt Asset', 'deck_link': 'https://aws.highspot.com/items/6773e1d0ae50e7759c31c662?lfrm=shp.1', 'demo_link': 'https://aws.highspot.com/items/6764dcfd156457a800fc4a9c?lfrm=shp.0', 'code_repo_link': 'https://aws.highspot.com/items/67af0cfd70ca740213815647?lfrm=shp.0', 'china_region_support': 'YES', 'industry': ['ALL'], 'team': 'SSA', 'contact': 'ybalbert@amazon.com', 'createtime': '2025-03-06 08:31:51', 'company': 'default', 'template': '', 'id': '1741249911185-9377787', 'demo_version': '2025v1'}, {'category': '翻译', 'demo_name': '图片翻译', 'description': '基于OCR对图像中文字进行检测，对检测到的boxing坐标通过Lama类SD模型进行内容感知擦除，使用LLM对检测到的文字进行翻译，翻译后的文字通过laystyle SD组件进行文字在指定区域的生成，最后合并擦除后的图像和文字图像层', 'demo_type': 'Simple Demo Asset', 'deck_link': 'https://aws.highspot.com/items/67c0475d52e91bea78df5e5a?lfrm=shp.1#1', 'demo_link': 'https://aws.highspot.co

/home/ec2-user/.local/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [15]:
# Upload the JSON data to a new DynamoDB table
upload_to_dynamodb(table_name, json_data)
print(f'uploded data from {filename}')

Data uploaded to DynamoDB table: prompt_hub_table
uploded data from GCR GenAI Asset Hub Management-0306.xlsx
